In [ ]:
import os
import glob
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.data.binning import get_bin_config
from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

fig_path = '/Users/harryclark/Documents/figs/FIGURE1/'
source_path = '/Users/harryclark/Downloads/COHORT12/'

cell_classifications = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
cell_classifications['SC_x'] = np.abs(cell_classifications['SC_x'])
cell_classifications


In [6]:
import numpy as np
df = cell_classifications.copy()
sc_bins = [(2800, 3000), (3000, 3200), (3200, 3400), (3400, 3600), (3600, 3800)]
cell_types = df['cell_type'].unique()

for cell_type in cell_types:
    scx = np.abs(df.loc[df['cell_type'] == cell_type, 'SC_x'].values)
    print(f'\n{cell_type}:')
    for (lo, hi) in sc_bins:
        in_bin = (scx >= lo) & (scx < hi)
        n_cells = in_bin.sum()
        if n_cells > 0:
            vals = scx[in_bin]
            n_unique = len(np.unique(vals))
            min_val, max_val = vals.min(), vals.max()
            print(f'  Bin {lo}-{hi}: {n_cells} cells, {n_unique} unique, range {min_val:.1f}–{max_val:.1f}')
        else:
            print(f'  Bin {lo}-{hi}: 0 cells')


GC:
  Bin 2800-3000: 26 cells, 4 unique, range 2850.0–2970.0
  Bin 3000-3200: 262 cells, 44 unique, range 3000.0–3199.6
  Bin 3200-3400: 170 cells, 19 unique, range 3200.2–3390.0
  Bin 3400-3600: 151 cells, 30 unique, range 3464.1–3540.0
  Bin 3600-3800: 91 cells, 51 unique, range 3600.0–3756.6

NG:
  Bin 2800-3000: 204 cells, 4 unique, range 2850.0–2970.0
  Bin 3000-3200: 806 cells, 67 unique, range 3000.0–3199.9
  Bin 3200-3400: 1029 cells, 38 unique, range 3200.1–3390.0
  Bin 3400-3600: 1737 cells, 108 unique, range 3465.4–3540.0
  Bin 3600-3800: 2012 cells, 315 unique, range 3600.0–3759.4

Other:
  Bin 2800-3000: 191 cells, 4 unique, range 2850.0–2970.0
  Bin 3000-3200: 346 cells, 34 unique, range 3000.0–3196.9
  Bin 3200-3400: 306 cells, 40 unique, range 3200.2–3390.0
  Bin 3400-3600: 669 cells, 57 unique, range 3464.1–3540.0
  Bin 3600-3800: 608 cells, 98 unique, range 3600.0–3759.4
